In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from google.cloud import storage

In [4]:
storage_client = storage.Client(project="foini-506814")

# LLM

Create a Google Cloud Platform project and activate Google Cloud SDK. 

Authenticate:

    gcloud auth application-default login

ATTENTION: you will get the model online and it has a cost!

In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    project="foini-506814",
    location="europe-west1",
    vertexai=True,
    temperature=0.9,
    max_tokens=1000,
    max_retries=1,
    timeout=120,
    include_thoughts=False,
)

### Example

In [6]:
messages = [
    SystemMessage(content="You are a helpful assistant that translates English to French. Translate the user sentence only."),
    HumanMessage(name="Pietro", content=[{"type": "text", "text": "I love programming."}]),
]

ai_msg = llm.invoke(messages)
ai_msg

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content="J'adore programmer.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0e2e3-87bb-7253-9666-f4949a4c7b7f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 5, 'total_tokens': 26, 'input_token_details': {'cache_read': 0}})

#### Multimodal

#### Public link

In [4]:
messages = [
    SystemMessage(content="You are a helpful assistant that recognize image content."),
    HumanMessage(name="Pietro", content=[
        {"type": "text", "text": "Describe the content of this image"},
        {"type": "image", "url": "https://static.desygner.com/wp-content/uploads/sites/13/2022/05/04150103/Free-Stock-Photos-03.jpg"},
    ]),
]

ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content="The image captures a dynamic moment of two young boys playing soccer in shallow water, likely during a misty sunrise or sunset. The scene is set outdoors, with a simple goal made of bamboo poles in the background. A soccer ball hangs in the air above one of the boys. On the left, a boy is captured mid-air, executing an acrobatic kick with impressive agility, sending splashes of water into the air. To his right, another boy stands, arms outstretched, appearing to be the goalkeeper ready to defend. The atmosphere is hazy and dreamlike, with a soft, warm light filtering through the mist, highlighting the water's surface and the figures. The lush greenery in the background is blurred, emphasizing the action in the foreground. The overall image evokes a sense of childhood innocence, energetic play, and a magical, serene environment.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provi

#### Storage

In [10]:
bucket = storage_client.bucket("chatbot-genius")
blob = bucket.blob("attachments/Résumé.pdf")
pdf_bytes = blob.download_as_bytes()

messages = [
    SystemMessage(
        content="You are a helpful assistant that analyzes files."
    ),
    HumanMessage(
        content=[
            {
                "type": "text",
                "text": "Describe the content of this file",
            },
            {
                "type": "media",
                "mime_type": "application/pdf",
                "data": pdf_bytes,
            },
        ]
    ),
]

ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content="This document is a resume for **Pietro Foini**, who identifies as a **Data Scientist**.\n\nHere's a summary of its content:\n\n*   **Contact Information:**\n    *   Phone: +39 3347029821\n    *   Email: pietro.foini1@gmail.com\n    *   Professional Links: GitHub (github.com/pietro-foini), LinkedIn (linkedin.com/in/pietro-foini), Google Scholar (scholar.google.com)\n\n*   **Summary:**\n    Pietro Foini is a Data Scientist with over 4 years of experience in transforming Machine Learning into practical, real-world solutions. His expertise spans the complete model lifecycle, including fine-tuning algorithms for Computer Vision, Graph Neural Networks (GNNs), forecasting, and Natural Language Processing (NLP), along with cloud deployment and performance tracking.\n\n*   **Experience:**\n    *   **NOVA AI TECHNOLOGY - ULIXE** (May 2022 - Present, Remote, Italy) - Data Scientist\n        *   Worked on projects for clients like Dale Consulting and Abstract", additional_kwargs

In [8]:
messages = [
    SystemMessage(
        content="You are a helpful assistant that analyzes files."
    ),
    HumanMessage(
        content=[
            {
                "type": "text",
                "text": "Describe the content of this file",
            },
            {
                "type": "media",
                "mime_type": "application/pdf",
                "file_uri": "gs://chatbot-genius/attachments/Résumé.pdf",
            },
        ]
    ),
]

ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content="This document is a resume for Pietro Foini, a Data Scientist.\n\nHere's a breakdown of the content:\n\n**Contact Information:**\n*   Name: Pietro Foini\n*   Title: Data Scientist\n*   Phone: +39 3347029821\n*   Email: pietro.foini1@gmail.com\n*   GitHub: github.com/pietro-foini\n*   LinkedIn: linkedin.com/in/pietro-foini\n*   Google Scholar: scholar.google.com\n\n**Summary:**\nPietro Foini is a Data Scientist with over 4 years of experience in Machine Learning, focusing on Computer Vision, GNNs, forecasting, and NLP. He enjoys the full model lifecycle, from fine-tuning to cloud deployment and performance tracking.\n\n**Experience:**\n*   **NOVA AI TECHNOLOGY - ULIXE (Data Scientist):** May 2022 - Present (Remote, Italy)\n    *   Worked with clients Dale Consulting and Abstract.it - EssilorLuxottica.\n    *   Led development of Machine Learning models for Anti-Money Laundering (AML) using GIANOS system and Graph Neural Networks (GNNs).\n    *   Managed end-to-end deve